# PlasMege experiments analysis
Cedric Chauve  

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import os
import datetime

print(datetime.date.today())

flexiblas BLAS backend  "AOCL" not found. Loading default (IMKL) instead.


2025-09-13


### Overview

We analyise the results of the experiments described in [README.md](file://README.md), and summarized below:
- for each sample, a set of plasmid bins files are given, either `ground_truth` or `mobrecon` or obtained by a combination of a plasmid classification method (`plasclass,plasgaph2,mlplasmids,rfplasmid`) and a plasmid binning method (`gplascc,plasbinflow`);  
- PlasMerge is applied to each such set of bins, with the same classification method used for binning and merging, with the caveat that `ground_truth` and `mobrecon` bins are merged once with each classfication method;
- PlasEval is used to compare accuracy statistics (`Precision,Recall,F1,Dissimilarity`) of unmerged and merged bins.  

**Important remark.** When running PlasEval, all sets of plasmid bns (including `ground_truth` bins) have contigs repeated within a bin beig deduplicated, in order to not penalize methods that do not account for multiplicity. However this creates an issue in the evaluation as when PlasMerge merges two bins having a common contig, only one copy is kept and if the `ground_truth` is composed of two bns, then this will result in a missing contig. Could PlasMerge decide, when jining bins, to duplicate a common contig based on coverage?

In [2]:
# Reading PlasEval results into dataframes
ANALYSIS_DIR = "analysis"

# PlasEval results with alpha=0.5
PLASEVAL_05_FILE = os.path.join(ANALYSIS_DIR, "plasmids_benchmarking_2025-08-02_data.filtered.randomized.plaseval.0.5.2025-09-12.csv")
RESULTS_05_DF = pd.read_csv(PLASEVAL_05_FILE)
PLASEVAL_00_FILE = os.path.join(ANALYSIS_DIR, "plasmids_benchmarking_2025-08-02_data.filtered.randomized.plaseval.0.0.2025-09-12.csv")
RESULTS_00_DF = pd.read_csv(PLASEVAL_00_FILE)

# Methods
CLASSIFICATION = ["plasclass", "plasgraph2", "mlplasmids", "rfplasmid"]
BINNING = ["ground_truth", "gplascc", "mobrecon", "plasbinflow"]
METHODS = [[b,c] for b in BINNING for c in CLASSIFICATION]

In [3]:
def find_inconsistencies(in_df):
    """
    Sanity check to ensure statistics between number of bins and PlasEval dissimilarity are consistent
    """
    for idx,row in in_df.iterrows():
        test1 = row["binning"] == "ground_truth"
        test2 = row["unmerged.nb_bins"] == row["merged.nb_bins"]
        test3 = row["unmerged.n.Dissimilarity"] != row["merged.n.Dissimilarity"]
        test4 = row["unmerged.nb_bins"] != row["merged.nb_bins"]
        test5 = row["unmerged.n.Dissimilarity"] == row["merged.n.Dissimilarity"]
        if test1 and ((test2 and test3) or (test4 and test5)):
            print(row)
            break
            
find_inconsistencies(RESULTS_05_DF)
find_inconsistencies(RESULTS_00_DF)

In [31]:
def filter_df(in_df, in_col, in_methods=None, zero=False, pos=False, neg=False):
    """
    Keep samples for which the result for in_col 
    - are present for both merged and unmerged bins,
    - show a difference that is either zero (zero=True), positive (pos=True) or negative (neg=True)
    """
    merged_col, unmerged_col = f"merged.{in_col}", f"unmerged.{in_col}"
    if in_methods is None:
        filtered_df = in_df.dropna(
            subset=[merged_col,unmerged_col]
        )
    else:
        filtered_df = in_df.loc[
            (in_df["classification"]==in_methods[1])
            &
            (in_df["binning"]==in_methods[0])
        ].dropna(
            subset=[merged_col,unmerged_col]
        )
    idx_list = []
    for idx,row in filtered_df.iterrows():
        diff = row[merged_col] - row[unmerged_col]
        if (zero and diff==0) or (pos and diff>0) or (neg and diff<0): 
            idx_list.append(idx)
    out_df = in_df.filter(items=idx_list, axis=0)
    return out_df

def summarize_col(in_df, in_col, in_methods):
    """
    Returns a dictionary providing a summary of the results for stat in_col:
    dictionary keys:
    - nb_samples: number samples for which we have the statistic for both unmerged and merged bins
    - nb_eq: number of samples for which the statistic is equal between merged and unmerged bins
    - nb_sup: number of samples for which the statistic is larger in merged bins than in unmerged bins
    - nb_inf: number of samples for which the statistic is smaller in merged bins than in unmerged bins
    - unmerged_total: sum of all values of the statistic in unmerged bins
    - merged_total: sum of all values of the statistic in merged bins
    - difference: merged_total - unmerged_total
"""
    merged_col, unmerged_col = f"merged.{in_col}", f"unmerged.{in_col}"
    zero_df = filter_df(in_df, in_col, in_methods=in_methods, zero=True)
    pos_df = filter_df(in_df, in_col, in_methods=in_methods, pos=True)
    neg_df = filter_df(in_df, in_col, in_methods=in_methods, neg=True)
    nb_zero = len(zero_df.index)
    nb_pos = len(pos_df.index)
    nb_neg = len(neg_df.index)
    nb_samples = nb_zero + nb_pos + nb_neg
    merged_total = zero_df[merged_col].sum() + pos_df[merged_col].sum() + neg_df[merged_col].sum()
    unmerged_total = zero_df[unmerged_col].sum() + pos_df[unmerged_col].sum() + neg_df[unmerged_col].sum()
    return {
        "nb_samples": nb_samples,
        "nb_eq": nb_zero,
        "nb_sup": nb_pos,
        "nb_inf": nb_neg,
        "merged_total": round(merged_total,2),
        "unmerged_total": round(unmerged_total,2),
        "difference": round(merged_total - unmerged_total,2)
    }

In [32]:
# Creating dictionaries used later to shw statistics

STATS_DICT = {}
for stat in ["nb_bins","nb_ctgs","len_ctgs","w.Precision", "w.Recall","w.F1","n.Dissimilarity","n.Cuts","n.Joins"]:
    STATS_DICT[stat] = {}
    STATS_DICT[stat]["aggregated"] = summarize_col(RESULTS_05_DF, stat, in_methods=None)
    for method in METHODS:
        method_key = f"{method[0]}.{method[1]}"
        STATS_DICT[stat][method_key] = summarize_col(RESULTS_05_DF, stat, in_methods=method)
        
for stat in ["u.Cuts", "u.Joins", "u.Extra_ctgs", "u.Missing_ctgs"]:
    STATS_DICT[stat] = {}
    STATS_DICT[stat]["aggregated"] = summarize_col(RESULTS_00_DF, stat, in_methods=None)
    for method in METHODS:
        method_key = f"{method[0]}.{method[1]}"
        STATS_DICT[stat][method_key] = summarize_col(RESULTS_00_DF, stat, in_methods=method)

STATS_DF = pd.DataFrame.from_dict(STATS_DICT, orient='index')

## Bins and contigs statistics

We first look at the number of bins -- that is expected to not increase in all samples --, number of contigs -- expected to increase.

In [34]:
NB_BINS_DF = pd.DataFrame.from_dict(STATS_DICT["nb_bins"], orient='index')
print("nb_sup: number of samples for which the statistics is larger in merged bins than in unmerged bins")
print("nb_inf: number of samples for which the statistics is smaller in merged bins than in unmerged bins")
print("nb_eq: number of samples for whic the statistics is equal in merged bins and in unmerged bins")
print("difference: merged_total - unmerged_total")
NB_BINS_DF 

nb_sup: number of samples for which the statistics is larger in merged bins than in unmerged bins
nb_inf: number of samples for which the statistics is smaller in merged bins than in unmerged bins
nb_eq: number of samples for whic the statistics is equal in merged bins and in unmerged bins
difference: merged_total - unmerged_total


,nb_samples,nb_eq,nb_sup,nb_inf,merged_total,unmerged_total,difference
aggregated,7806,3264,0,4542,28898.0,61087,-32189.0
ground_truth.plasclass,498,362,0,136,1310.0,1509,-199.0
ground_truth.plasgraph2,497,330,0,167,1225.0,1487,-262.0
ground_truth.mlplasmids,498,311,0,187,1164.0,1509,-345.0
ground_truth.rfplasmid,498,310,0,188,1159.0,1509,-350.0
gplascc.plasclass,436,8,0,428,3462.0,8645,-5183.0
gplascc.plasgraph2,464,70,0,394,2361.0,5598,-3237.0
gplascc.mlplasmids,442,5,0,437,3609.0,14667,-11058.0
gplascc.rfplasmid,495,224,0,271,1706.0,2454,-748.0
mobrecon.plasclass,496,386,0,110,1739.0,1944,-205.0


We can observe that merging has a significant effect on the number of bins, going from a little more than $60,000$ before merging to less than $30,000$ after merging.  

First, an important comment is that for `ground_truth` bins, then PlasMerge does merge bins in a significant number of cases; this casts a shadow on the expected accuracy of the merged bins, as by definition merged bins from `ground_tuth` bins are inaccurate.

Merging was very aggessive on bins obtained with `gplascc` and `plasbin-flow`. But for `gplascc` asid of the combination `gplascc+rfplasmid`, one can observe a very large number of bins prior to merging, that shows a high-level of fragmentation; it will be iteresting to see if the large number of mergings will be accurate.

**Imporant comment.**  It also raises the question of the closenes of plasmids in the assembly graph, that is necessary for two `ground_truth` bins to be merged. This question has not been studied in benchmarking papers (it wa one of the questions asked to Mohammad), it is cental for the basic idea of PlasMerge, but also for classification methods (does motivate a GNN model) and binning (Due to the increased risk of ixing plasmid bins).

In [35]:
NB_CTGS_DF = pd.DataFrame.from_dict(STATS_DICT["nb_ctgs"], orient='index')
NB_CTGS_DF 

,nb_samples,nb_eq,nb_sup,nb_inf,merged_total,unmerged_total,difference
aggregated,7806,5704,0,2102,280690.0,439396,-158706.0
ground_truth.plasclass,498,391,0,107,16031.0,16719,-688.0
ground_truth.plasgraph2,497,367,0,130,15767.0,16684,-917.0
ground_truth.mlplasmids,498,346,0,152,15454.0,16719,-1265.0
ground_truth.rfplasmid,498,348,0,150,15434.0,16719,-1285.0
gplascc.plasclass,436,310,0,126,23889.0,24423,-534.0
gplascc.plasgraph2,464,370,0,94,13984.0,14367,-383.0
gplascc.mlplasmids,442,345,0,97,27734.0,28105,-371.0
gplascc.rfplasmid,495,421,0,74,13032.0,13289,-257.0
mobrecon.plasclass,496,496,0,0,9211.0,9211,0.0


**Remark.** The number of contigs does never increase, despite PlasMerge being supposed to join bins through paths. That in some cases the number of contigs does decrease can be explained by the above remark on PlasMerge not duplicating contigs when merging two overlapping bins. But I do not understand that we see no case where the number of contigs does increase. 

## PlasEval dissimilarity

We now look at the normalized dissimilarity score (with `alpha=0.5`) to assess accuracy.

In [36]:
DISSIMILARITY_DF = pd.DataFrame.from_dict(STATS_DICT["n.Dissimilarity"], orient='index')
DISSIMILARITY_DF 

,nb_samples,nb_eq,nb_sup,nb_inf,merged_total,unmerged_total,difference
aggregated,7530,3848,1444,2238,2307.31,2373.58,-66.27
ground_truth.plasclass,498,362,136,0,10.94,0.00,10.94
ground_truth.plasgraph2,496,330,166,0,14.94,0.00,14.94
ground_truth.mlplasmids,498,311,187,0,18.73,0.00,18.73
ground_truth.rfplasmid,497,310,187,0,18.68,0.00,18.68
gplascc.plasclass,432,92,70,270,206.89,210.25,-3.36
gplascc.plasgraph2,463,168,63,232,231.13,235.06,-3.92
gplascc.mlplasmids,436,72,73,291,220.01,224.96,-4.94
gplascc.rfplasmid,494,282,63,149,174.84,175.73,-0.89
mobrecon.plasclass,496,415,56,25,162.07,160.65,1.42


Overall we see a very slight decrease in the dissimilarity, which is what we want, but it is very minimal. Wen looking on a per-sample bases, for more samples do we see a decrease in dissimilarity, which is also good.   
Obviously for `ground_truth` samples, the dissimilarity alway increase.  
Interestingly, for `mobrecon`, the most precise method, the similarity does increase very slightly after merging and there are more samples for which th dissimilarity increases than fo which the dissimilarity does decrease. This raises a similar question than when looking at `ground_truth` about the closeness of plasmids in the assembly graph.  
The most important decrease of dissimilarity is for `plasbinflow`.  

But overall, when coparing the various binning methods (excluding `ground_truth`) PlasMerge has a minimal impact and does not bring `gplascc` an `plasbinflow` close to `mobrecon`.


## PlasEval cuts and joins

PlasMerge doing joins on bins, we look at how often they are accurate, by looking a the number of cuts and joins reported by PlasEval (so with `alpha=0` to not weight them by the length of contigs). If joins done by PlasMerge are correct, then we expect that
- the number of joins needed by PlasEval is smaller with merged bins than with unmerged bins;
- the number of cuts needed by PlasEval is not greater with merged bins (to correct improper mergings) than with unmerged bins.

In [37]:
JOINS_DF = pd.DataFrame.from_dict(STATS_DICT["u.Joins"], orient='index')
JOINS_DF 

,nb_samples,nb_eq,nb_sup,nb_inf,merged_total,unmerged_total,difference
aggregated,7668,5427,0,2241,3595.0,10415.0,-6820.0
ground_truth.plasclass,498,498,0,0,0.0,0.0,0.0
ground_truth.plasgraph2,497,497,0,0,0.0,0.0,0.0
ground_truth.mlplasmids,498,498,0,0,0.0,0.0,0.0
ground_truth.rfplasmid,498,498,0,0,0.0,0.0,0.0
gplascc.plasclass,432,190,0,242,545.0,1488.0,-943.0
gplascc.plasgraph2,464,224,0,240,347.0,1387.0,-1040.0
gplascc.mlplasmids,437,161,0,276,347.0,1833.0,-1486.0
gplascc.rfplasmid,494,332,0,162,351.0,710.0,-359.0
mobrecon.plasclass,496,462,0,34,399.0,446.0,-47.0


In [38]:
CUTS_DF = pd.DataFrame.from_dict(STATS_DICT["u.Cuts"], orient='index')
CUTS_DF

,nb_samples,nb_eq,nb_sup,nb_inf,merged_total,unmerged_total,difference
aggregated,7668,6094,1488,86,4871.0,2798.0,2073.0
ground_truth.plasclass,498,370,128,0,188.0,0.0,188.0
ground_truth.plasgraph2,497,337,160,0,253.0,0.0,253.0
ground_truth.mlplasmids,498,317,181,0,334.0,0.0,334.0
ground_truth.rfplasmid,498,315,183,0,339.0,0.0,339.0
gplascc.plasclass,432,379,45,8,277.0,235.0,42.0
gplascc.plasgraph2,464,413,43,8,277.0,233.0,44.0
gplascc.mlplasmids,437,379,47,11,247.0,208.0,39.0
gplascc.rfplasmid,494,428,61,5,344.0,282.0,62.0
mobrecon.plasclass,496,452,44,0,367.0,316.0,51.0


Overall, this is positive. Aside of the `ground_truth` case obviously, the number of cuts does increase in merged bins, which means some joins need to be corrected, but this increase is smaller than the decrease in the number of joins, so more joins are correct than erroneous. This is very significant for `gplascc` and `plasbinflow`. 

Looking at the normalized cuts and joins to see their impact on the PlasEval dissimilarity score.

In [39]:
NJOINS_DF = pd.DataFrame.from_dict(STATS_DICT["n.Joins"], orient='index')
NJOINS_DF 

,nb_samples,nb_eq,nb_sup,nb_inf,merged_total,unmerged_total,difference
aggregated,7530,5328,89,2113,70.41,152.03,-81.62
ground_truth.plasclass,498,498,0,0,0.00,0.00,0.00
ground_truth.plasgraph2,496,496,0,0,0.00,0.00,0.00
ground_truth.mlplasmids,498,498,0,0,0.00,0.00,0.00
ground_truth.rfplasmid,497,497,0,0,0.00,0.00,0.00
gplascc.plasclass,432,181,8,243,5.64,10.43,-4.79
gplascc.plasgraph2,463,220,4,239,4.88,10.69,-5.81
gplascc.mlplasmids,436,157,2,277,5.05,12.25,-7.20
gplascc.rfplasmid,494,329,1,164,8.20,11.99,-3.79
mobrecon.plasclass,496,453,3,40,9.42,10.39,-0.97


In [40]:
NCUTS_DF = pd.DataFrame.from_dict(STATS_DICT["n.Cuts"], orient='index')
NCUTS_DF

,nb_samples,nb_eq,nb_sup,nb_inf,merged_total,unmerged_total,difference
aggregated,7530,5526,1987,17,160.12,57.94,102.18
ground_truth.plasclass,498,370,128,0,7.56,0.00,7.56
ground_truth.plasgraph2,496,337,159,0,11.04,0.00,11.04
ground_truth.mlplasmids,498,317,181,0,13.56,0.00,13.56
ground_truth.rfplasmid,497,315,182,0,13.52,0.00,13.52
gplascc.plasclass,432,320,109,3,8.54,6.29,2.25
gplascc.plasgraph2,463,356,106,1,7.76,5.74,2.02
gplascc.mlplasmids,436,311,122,3,8.33,5.95,2.39
gplascc.rfplasmid,494,388,104,2,11.85,8.78,3.07
mobrecon.plasclass,496,437,59,0,8.57,6.19,2.39


Overall, the extra cuts needed to correct erroneous joins ave a greater impact o the dissimilarity than the correct joins done by PlasMerge.  

## Precision, Recall, F1

Last we look at (weighted) precision, recall and F1 as these are statistics independent of the PlasEval model.

In [30]:
F1_DF = pd.DataFrame.from_dict(STATS_DICT["w.F1"], orient='index')
F1_DF

,nb_samples,nb_eq,nb_sup,nb_inf,merged_total,unmerged_total,difference
aggregated,7806,4205,1571,2030,5376.37,5531.19,154.82
ground_truth.plasclass,498,370,0,128,470.50,489.00,18.50
ground_truth.plasgraph2,497,337,0,160,460.30,488.00,27.70
ground_truth.mlplasmids,498,317,0,181,453.43,489.00,35.57
ground_truth.rfplasmid,498,315,0,183,453.80,489.00,35.20
gplascc.plasclass,436,163,165,108,236.15,238.74,2.59
gplascc.plasgraph2,464,212,185,67,227.99,228.27,0.28
gplascc.mlplasmids,442,134,217,91,241.90,242.36,0.46
gplascc.rfplasmid,495,301,105,89,318.44,320.71,2.27
mobrecon.plasclass,496,428,13,55,361.72,365.64,3.92


This tells a similar story than the PlasEval dissimilarity, of a mixed bag of results.

In [27]:
PRECISION_DF = pd.DataFrame.from_dict(STATS_DICT["w.Precision"], orient='index')
PRECISION_DF

,nb_samples,nb_eq,nb_sup,nb_inf,merged_total,unmerged_total,difference
aggregated,7806,4956,437,2413,5312.21,5721.37,409.16
ground_truth.plasclass,498,370,0,128,458.55,489.00,30.45
ground_truth.plasgraph2,497,337,0,160,443.28,488.00,44.72
ground_truth.mlplasmids,498,317,0,181,431.98,489.00,57.02
ground_truth.rfplasmid,498,315,0,183,432.58,489.00,56.42
gplascc.plasclass,436,256,26,154,201.55,210.24,8.69
gplascc.plasgraph2,464,325,18,121,279.53,288.97,9.44
gplascc.mlplasmids,442,269,33,140,234.07,244.89,10.82
gplascc.rfplasmid,495,370,11,114,322.19,333.15,10.96
mobrecon.plasclass,496,438,0,58,368.94,376.97,8.04


In [29]:
RECALL_DF = pd.DataFrame.from_dict(STATS_DICT["w.Recall"], orient='index')
RECALL_DF

,nb_samples,nb_eq,nb_sup,nb_inf,merged_total,unmerged_total,difference
aggregated,7806,5422,2384,0,6407.52,6196.18,-211.34
ground_truth.plasclass,498,498,0,0,489.00,489.00,0.00
ground_truth.plasgraph2,497,497,0,0,488.00,488.00,0.00
ground_truth.mlplasmids,498,498,0,0,489.00,489.00,0.00
ground_truth.rfplasmid,498,498,0,0,489.00,489.00,0.00
gplascc.plasclass,436,195,241,0,351.29,340.75,-10.55
gplascc.plasgraph2,464,230,234,0,238.34,232.23,-6.11
gplascc.mlplasmids,442,167,275,0,335.37,323.88,-11.50
gplascc.rfplasmid,495,328,167,0,369.07,361.99,-7.09
mobrecon.plasclass,496,455,41,0,405.75,403.49,-2.25
